# 04 - Final Evaluation & Explainability

Evaluates all trained models, generates figures, saves the comparison table, tunes the decision threshold, and prints the final classification report.

In [1]:
import os
import sys
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
sys.path.insert(0, ROOT)
MODELS = os.path.join(ROOT, 'models')

from src.evaluate import (
    load_test_data, evaluate_model, plot_confusion_matrices, plot_roc_curves,
    plot_precision_recall_curves, plot_feature_importance, plot_shap_summary,
    save_comparison_table,
)
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

## 1. Load test data and all models

In [2]:
X_test, y_test = load_test_data()
loaded = {}
for name in ['logistic_regression', 'random_forest', 'xgboost_model']:
    with open(os.path.join(MODELS, f'{name}.pkl'), 'rb') as f:
        loaded[name] = pickle.load(f)
results = [evaluate_model(m, X_test, y_test, n) for n, m in loaded.items()]

C:\Users\krish\miniconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\krish\miniconda3\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


## 2. Generate all figures

In [3]:
plot_confusion_matrices(results, y_test)
plot_roc_curves(results, y_test)
plot_precision_recall_curves(results, y_test)
plot_feature_importance(loaded['random_forest'], X_test.columns.tolist())
plot_feature_importance(loaded['xgboost_model'], X_test.columns.tolist())
plot_shap_summary(loaded['xgboost_model'], X_test, model_name='xgboost_model')
print('Figures written to reports/figures/')

Figures written to reports/figures/


## 3. Save model comparison table

In [4]:
comparison = save_comparison_table(results)
comparison


Model Comparison:
              Model  Accuracy       F1  ROC_AUC  Avg_Precision
logistic_regression  0.973596 0.106888 0.970417       0.728574
      random_forest  0.999087 0.763636 0.983780       0.856554
      xgboost_model  0.999157 0.777778 0.979548       0.865661


,Model,Accuracy,F1,ROC_AUC,Avg_Precision
0,logistic_regression,0.973596,0.106888,0.970417,0.728574
1,random_forest,0.999087,0.763636,0.983780,0.856554
2,xgboost_model,0.999157,0.777778,0.979548,0.865661


## 4. Threshold tuning (precision / recall tradeoff)

In [5]:
best_name = comparison.loc[comparison['ROC_AUC'].idxmax(), 'Model']
y_prob = loaded[best_name].predict_proba(X_test)[:, 1]
thresholds = np.arange(0.1, 0.91, 0.05)
rows = []
for t in thresholds:
    y_hat = (y_prob >= t).astype(int)
    rows.append({'threshold': round(t, 2),
                 'precision': precision_score(y_test, y_hat, zero_division=0),
                 'recall': recall_score(y_test, y_hat),
                 'f1': f1_score(y_test, y_hat)})
tune = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 5))
for col in ['precision', 'recall', 'f1']:
    ax.plot(tune['threshold'], tune[col], marker='o', label=col)
ax.set_xlabel('Decision Threshold'); ax.set_ylabel('Score')
ax.set_title(f'Threshold Tuning - {best_name}'); ax.legend()
plt.tight_layout(); plt.show()
tune

C:\Users\krish\AppData\Local\Temp\ipykernel_30932\1745530008.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


,threshold,precision,recall,f1
0,0.10,0.081008,0.918367,0.148883
1,0.15,0.139498,0.908163,0.241848
2,0.20,0.231771,0.908163,0.369295
3,0.25,0.317690,0.897959,0.469333
4,0.30,0.396396,0.897959,0.550000
5,0.35,0.470270,0.887755,0.614841
6,0.40,0.537975,0.867347,0.664062
7,0.45,0.622222,0.857143,0.721030
8,0.50,0.688525,0.857143,0.763636
9,0.55,0.713043,0.836735,0.769953


## 5. Final classification report for the best model

In [6]:
y_pred = loaded[best_name].predict(X_test)
print(f'Best model: {best_name}\n')
print(classification_report(y_test, y_pred))

Best model: random_forest

              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.69      0.86      0.76        98

    accuracy                           1.00     56962
   macro avg       0.84      0.93      0.88     56962
weighted avg       1.00      1.00      1.00     56962

